# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

# Reading from Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')

# Data Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Normalization and Standardization

In [0]:
df = (
    df
    .withColumn(
        'GEN', 
        F.when(col('GEN').isNull(), 'Unknown')
        .when(F.upper(col('GEN')) == 'M', 'Male')
        .when(F.upper(col('GEN')) == 'F', 'Female')
        .otherwise(col('GEN'))
    ))

### Making the 'CID' useful to connect with other tables

In [0]:
df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

### Birthdate Validation

In [0]:
df = df.withColumn(
    'bdate',
    F.when(col('bdate') > F.current_date(), None)
    .otherwise(col('bdate'))
)

# Renaming the Columns

In [0]:
renaming_map = {
    'cid': 'customer_number',
    'bdate': 'birth_date',
    'gen': 'gender'
}

for old_name, new_name in renaming_map.items():
    df = df.withColumnRenamed(old_name, new_name)

# Checking the DataFrame

In [0]:
df.limit(10).display()

# Writing into Silver Table

In [0]:
(
    df.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable('workspace.silver.erp_customers')
)